# Dedicated Clustering Preprocessing Pipeline
**Dataset**: Airplane Customer Satisfaction (`data/raw/airplane customer satisfaction.csv`)

This notebook performs complete, leak-free preprocessing tailored specifically for unsupervised customer segmentation.

## SECTION 1 — LOAD DATA

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set relative path to raw dataset
RAW_DATA_PATH = "../../data/raw/airplane customer satisfaction.csv"

# Load dataset
df_raw = pd.read_csv(RAW_DATA_PATH)

print("=== RAW DATASET AUDIT ===")
print("Shape:", df_raw.shape)
print("\nFirst 5 rows:")
display(df_raw.head())

print("\nData Types:")
print(df_raw.dtypes)

print("\nDescriptive Statistics:")
display(df_raw.describe(include="all").T)

=== RAW DATASET AUDIT ===
Shape: (103904, 25)

First 5 rows:
   Unnamed: 0      id  ... Arrival Delay in Minutes             satisfaction
0           0   70172  ...                     18.0  neutral or dissatisfied
1           1    5047  ...                      6.0  neutral or dissatisfied
2           2  110028  ...                      0.0                satisfied
3           3   24026  ...                      9.0  neutral or dissatisfied
4           4  119299  ...                      0.0                satisfied

[5 rows x 25 columns]

Data Types:
Unnamed: 0                             int64
id                                     int64
Gender                                   str
Customer Type                            str
Age                                    int64
Type of Travel                           str
Class                                    str
Flight Distance                        int64
Inflight wifi service                  int64
Departure/Arrival time convenient   

## SECTION 2 — DATA QUALITY AUDIT

In [4]:
print("=== DATA QUALITY AUDIT ===")

# Missing values
missing = df_raw.isnull().sum()
missing_cols = missing[missing > 0]
print("Missing values per column:")
print(missing_cols if len(missing_cols) > 0 else "No missing values.")

# Duplicate rows
duplicate_count = df_raw.duplicated().sum()
print(f"\nDuplicate rows: {duplicate_count}")

# Unique value counts
print("\nUnique value counts per column:")
for col in df_raw.columns:
    print(f"  {col}: {df_raw[col].nunique()} unique values")

# Constant / zero-variance columns
constant_cols = [col for col in df_raw.columns if df_raw[col].nunique() <= 1]
print(f"\nConstant (zero-variance) columns: {constant_cols}")

=== DATA QUALITY AUDIT ===
Missing values per column:
Arrival Delay in Minutes    310
dtype: int64

Duplicate rows: 0

Unique value counts per column:
  Unnamed: 0: 103904 unique values
  id: 103904 unique values
  Gender: 2 unique values
  Customer Type: 2 unique values
  Age: 75 unique values
  Type of Travel: 2 unique values
  Class: 3 unique values
  Flight Distance: 3802 unique values
  Inflight wifi service: 6 unique values
  Departure/Arrival time convenient: 6 unique values
  Ease of Online booking: 6 unique values
  Gate location: 6 unique values
  Food and drink: 6 unique values
  Online boarding: 6 unique values
  Seat comfort: 6 unique values
  Inflight entertainment: 6 unique values
  On-board service: 6 unique values
  Leg room service: 6 unique values
  Baggage handling: 5 unique values
  Checkin service: 6 unique values
  Inflight service: 6 unique values
  Cleanliness: 6 unique values
  Departure Delay in Minutes: 446 unique values
  Arrival Delay in Minutes: 455 uniqu

## SECTION 3 — REMOVE NON-FEATURE COLUMNS & EXCLUDE TARGET

In [6]:
# Create working copy
df = df_raw.copy()

# Remove ID / index columns
id_cols = ["Unnamed: 0", "id"]
df = df.drop(columns=[col for col in id_cols if col in df.columns])
print(f"Dropped ID columns: {id_cols}")

# Exclude target/label column 'satisfaction'
TARGET_COL = "satisfaction"
if TARGET_COL in df.columns:
    y_optional = df[TARGET_COL].copy()
    df = df.drop(columns=[TARGET_COL])
    print(f"Excluded target column '{TARGET_COL}' from feature matrix. Preserved as optional post-hoc label.")
else:
    y_optional = None

print(f"Remaining dataset shape: {df.shape}")

Dropped ID columns: ['Unnamed: 0', 'id']
Excluded target column 'satisfaction' from feature matrix. Preserved as optional post-hoc label.
Remaining dataset shape: (103904, 22)


## SECTION 4 — FEATURE IDENTIFICATION

In [8]:
# Identify feature categories
nominal_categorical_cols = ["Gender", "Customer Type", "Type of Travel", "Class"]
ordinal_survey_cols = [
    "Inflight wifi service", "Departure/Arrival time convenient", "Ease of Online booking",
    "Gate location", "Food and drink", "Online boarding", "Seat comfort",
    "Inflight entertainment", "On-board service", "Leg room service",
    "Baggage handling", "Checkin service", "Inflight service", "Cleanliness"
]
continuous_num_cols = ["Age", "Flight Distance", "Departure Delay in Minutes", "Arrival Delay in Minutes"]

print("=== FEATURE TYPE CLASSIFICATION ===")
print(f"Nominal Categorical ({len(nominal_categorical_cols)}): {nominal_categorical_cols}")
print(f"Ordinal Survey Ratings ({len(ordinal_survey_cols)}): {ordinal_survey_cols}")
print(f"Continuous Numerical ({len(continuous_num_cols)}): {continuous_num_cols}")

=== FEATURE TYPE CLASSIFICATION ===
Nominal Categorical (4): ['Gender', 'Customer Type', 'Type of Travel', 'Class']
Ordinal Survey Ratings (14): ['Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Inflight service', 'Cleanliness']
Continuous Numerical (4): ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']


## SECTION 5 — MISSING VALUE HANDLING

In [10]:
# Check missing values before imputation
missing_arrival_delay = df["Arrival Delay in Minutes"].isnull().sum()
print(f"Missing values in 'Arrival Delay in Minutes' before imputation: {missing_arrival_delay}")

# Median imputation rationale: Flight delay durations are heavily right-skewed with extreme long-tail outliers.
# Median is robust against extreme outlier distortion.
median_val = df["Arrival Delay in Minutes"].median()
df["Arrival Delay in Minutes"] = df["Arrival Delay in Minutes"].fillna(median_val)

print(f"Imputed missing values using median = {median_val:.1f}")
assert df["Arrival Delay in Minutes"].isnull().sum() == 0, "Error: Missing values remain!"
print("Verification: Zero missing values remain in the dataset.")

Missing values in 'Arrival Delay in Minutes' before imputation: 310
Imputed missing values using median = 0.0
Verification: Zero missing values remain in the dataset.


## SECTION 6 — OUTLIER AND DISTRIBUTION ANALYSIS

In [12]:
# Calculate skewness for continuous numerical variables
skew_df = []
continuous_vars = ["Flight Distance", "Departure Delay in Minutes", "Arrival Delay in Minutes"]

for col in continuous_vars:
    skew_val = df[col].skew()
    if skew_val > 1.0:
        transform = "log1p"
        reason = f"Right-skewed continuous feature (skewness = {skew_val:.2f} > 1.0)"
    else:
        transform = "None"
        reason = f"Unskewed/moderately skewed feature (skewness = {skew_val:.2f})"
    skew_df.append({
        "Feature": col,
        "Skewness Before": round(skew_val, 2),
        "Transformation": transform,
        "Reason": reason
    })

transformation_summary = pd.DataFrame(skew_df)
print("=== TRANSFORMATION SUMMARY ===")
display(transformation_summary)

# Apply log1p to justified continuous features
for row in skew_df:
    if row["Transformation"] == "log1p":
        col = row["Feature"]
        df[col] = np.log1p(df[col])
        print(f"Applied log1p to '{col}'. Skewness after: {df[col].skew():.2f}")

=== TRANSFORMATION SUMMARY ===
                      Feature  ...                                             Reason
0             Flight Distance  ...  Right-skewed continuous feature (skewness = 1....
1  Departure Delay in Minutes  ...  Right-skewed continuous feature (skewness = 6....
2    Arrival Delay in Minutes  ...  Right-skewed continuous feature (skewness = 6....

[3 rows x 4 columns]
Applied log1p to 'Flight Distance'. Skewness after: -0.20
Applied log1p to 'Departure Delay in Minutes'. Skewness after: 0.92
Applied log1p to 'Arrival Delay in Minutes'. Skewness after: 0.88


## SECTION 7 — CATEGORICAL ENCODING

In [14]:
# One-hot encode nominal categorical variables with drop_first=True
print("Nominal categorical features to encode:", nominal_categorical_cols)
print(f"Shape before one-hot encoding: {df.shape}")

df_encoded = pd.get_dummies(df, columns=nominal_categorical_cols, drop_first=True, dtype=float)
print(f"Shape after one-hot encoding: {df_encoded.shape}")

new_encoded_cols = [c for c in df_encoded.columns if any(c.startswith(cat + "_") for cat in nominal_categorical_cols)]
print(f"\nGenerated One-Hot Encoded features ({len(new_encoded_cols)}):")
print(new_encoded_cols)

Nominal categorical features to encode: ['Gender', 'Customer Type', 'Type of Travel', 'Class']
Shape before one-hot encoding: (103904, 22)
Shape after one-hot encoding: (103904, 23)

Generated One-Hot Encoded features (5):
['Gender_Male', 'Customer Type_disloyal Customer', 'Type of Travel_Personal Travel', 'Class_Eco', 'Class_Eco Plus']


## SECTION 8 — FEATURE SCALING

In [16]:
from sklearn.preprocessing import StandardScaler

# Scale all features using StandardScaler
# Rationale: Fitted on full dataset as unsupervised clustering does not perform train/test splitting.
scaler = StandardScaler()
X_scaled_array = scaler.fit_transform(df_encoded)

X_scaled = pd.DataFrame(X_scaled_array, columns=df_encoded.columns)

print("=== FEATURE SCALING COMPLETED ===")
print("Scaled feature matrix shape:", X_scaled.shape)
print("Max mean across scaled features:", np.abs(X_scaled.mean(axis=0)).max())
print("Max std deviation difference from 1.0:", np.abs(X_scaled.std(axis=0) - 1.0).max())

=== FEATURE SCALING COMPLETED ===
Scaled feature matrix shape: (103904, 23)
Max mean across scaled features: 1.9311794404320554e-16
Max std deviation difference from 1.0: 4.81216901326853e-06


## SECTION 9 — FINAL VALIDATION

In [18]:
print("=== FINAL MATRIX VALIDATION ===")
print(f"Final shape: {X_scaled.shape}")
print(f"All columns numeric: {all(np.issubdtype(dt, np.number) for dt in X_scaled.dtypes)}")
print(f"Total NaN count: {X_scaled.isnull().sum().sum()}")
print(f"Total Inf count: {np.isinf(X_scaled.values).sum()}")

# Check constant features
constant_features = [c for c in X_scaled.columns if X_scaled[c].std() == 0]
print(f"Constant features count: {len(constant_features)}")

# Check ID / label leak
assert "Unnamed: 0" not in X_scaled.columns and "id" not in X_scaled.columns, "ID columns present!"
assert "satisfaction" not in X_scaled.columns, "Target column 'satisfaction' present!"
print("Validation passed successfully! No ID columns or target labels present in X.")

=== FINAL MATRIX VALIDATION ===
Final shape: (103904, 23)
All columns numeric: True
Total NaN count: 0
Total Inf count: 0
Constant features count: 0
Validation passed successfully! No ID columns or target labels present in X.


## SECTION 10 — SAVE PROCESSED DATA

In [20]:
OUTPUT_DIR = "../../data/processed/clustering"
os.makedirs(OUTPUT_DIR, exist_ok=True)

X_OUTPUT_PATH = os.path.join(OUTPUT_DIR, "X.csv")
Y_OUTPUT_PATH = os.path.join(OUTPUT_DIR, "satisfaction.csv")

# Save scaled feature matrix X
X_scaled.to_csv(X_OUTPUT_PATH, index=False)
print(f"Saved preprocessed feature matrix X ({X_scaled.shape}) to: {X_OUTPUT_PATH}")

# Save optional target labels for post-hoc analysis
if y_optional is not None:
    y_optional.to_frame(name="satisfaction").to_csv(Y_OUTPUT_PATH, index=False)
    print(f"Saved optional satisfaction labels ({y_optional.shape}) to: {Y_OUTPUT_PATH}")

Saved preprocessed feature matrix X ((103904, 23)) to: ../../data/processed/clustering\X.csv
Saved optional satisfaction labels ((103904,)) to: ../../data/processed/clustering\satisfaction.csv
